In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Ayarlar
DATA_PATH = r"C:\Users\Dogan\Desktop\Ayberk Dosyalar\Bitirme Projesi\Fundus\all_fundus_datas\vessel_segment_yetiskin_data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
IMG_SIZE = (512, 512) # Bellek yönetimi için makaledeki gibi yamalamak yerine resize ediyoruz
LEARNING_RATE = 1e-4
EPOCHS = 50

class FundusAVDataset(Dataset):
    def __init__(self, data_path, split_file, transform=None):
        self.data_path = data_path
        self.transform = transform
        
        # Görüntü isimlerini txt dosyasından oku [cite: 151]
        with open(os.path.join(data_path, split_file), 'r') as f:
            self.image_names = [line.strip() for line in f.readlines()]

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.data_path, "images", img_name)
        mask_path = os.path.join(self.data_path, "annotation", img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        # Renkli maskeyi sınıf indekslerine dönüştür [cite: 86, 104]
        mask = np.zeros((mask_rgb.shape[0], mask_rgb.shape[1]), dtype=np.long)
        
        # Renk eşleşmeleri
        mask[(mask_rgb == [255, 0, 0]).all(axis=-1)] = 1 # Arter
        mask[(mask_rgb == [0, 0, 255]).all(axis=-1)] = 2 # Ven
        mask[(mask_rgb == [0, 255, 0]).all(axis=-1)] = 3 # Kesişim (Crossing)
        mask[(mask_rgb == [255, 255, 255]).all(axis=-1)] = 4 # Belirsiz

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask



train_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_ds = FundusAVDataset(DATA_PATH, "training.txt", transform=train_transform)
val_ds = FundusAVDataset(DATA_PATH, "testing.txt", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


# --- GÜNCEL 4. HÜCRE: GÜÇLÜ MİMARİ ---
import segmentation_models_pytorch as smp

# 1. Mimari Seçimi: Unet++ ve EfficientNet-B5
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5", 
    encoder_weights="imagenet",     # Pretrained ağırlıklar
    in_channels=3,
    classes=5,
).to(DEVICE)



# Arter ve Ven odaklı olduğu için DiceLoss ve CrossEntropy kombinasyonu iyidir
criterion = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


# --- 5. HÜCRE GÜNCELLEME ---
best_iou = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        # DÜZELTME: .long() ekledik
        images, masks = images.to(DEVICE), masks.to(DEVICE).long()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation kısmında da aynı düzeltmeyi yapıyoruz
    model.eval()
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            # DÜZELTME: .long() ekledik
            images, masks = images.to(DEVICE), masks.to(DEVICE).long()
            outputs = model(images)
            
            preds = torch.argmax(outputs, dim=1)
            stats = smp.metrics.get_stats(preds, masks, mode='multiclass', num_classes=5)
            tp += stats[0]; fp += stats[1]; fn += stats[2]; tn += stats[3]

    current_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro-imagewise").item()
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val IoU: {current_iou:.4f}")

    if current_iou > best_iou:
        best_iou = current_iou
        torch.save(model.state_dict(), "best_fundus_model_efficient.pth")
        print("--> Model başarıyla güncellendi ve kaydedildi!")